# AP31 - FastAPI for AI Applications

### Build production-ready APIs for your AI models

**Duration:** 3 hours

**What you'll learn:**
- Building AI APIs with FastAPI
- Request/response models with Pydantic
- Streaming responses for LLMs
- WebSocket for real-time communication
- Authentication and rate limiting
- Error handling and logging
- Deployment best practices

**Prerequisites:**
- Completed AP01-AP03 (LlamaIndex basics) OR AG01-AG06 (LangChain basics)
- Understanding of REST APIs
- Basic async Python

---

## 🔧 Setup

In [ ]:
# Install dependencies
# !pip install -qU fastapi uvicorn[standard] python-multipart \
#     llama-index llama-index-llms-openai python-dotenv pydantic slowapi python-jose[cryptography]

---

## 1. Hello World FastAPI + AI

In [ ]:
%%writefile app_basic.py
from fastapi import FastAPI
from pydantic import BaseModel
from llama_index.core import VectorStoreIndex, Document
import os
from dotenv import load_dotenv

load_dotenv()

# Initialize FastAPI
app = FastAPI(
    title="AI Q&A API",
    description="Simple AI-powered question answering",
    version="1.0.0"
)

# Create simple knowledge base
docs = [
    Document(text="FastAPI is a modern Python web framework for building APIs."),
    Document(text="LlamaIndex is a data framework for LLM applications.")
]
index = VectorStoreIndex.from_documents(docs)
query_engine = index.as_query_engine()

# Request/Response models
class QuestionRequest(BaseModel):
    question: str
    
class AnswerResponse(BaseModel):
    question: str
    answer: str

# Endpoints
@app.get("/")
async def root():
    return {"message": "AI Q&A API is running!"}

@app.post("/ask", response_model=AnswerResponse)
async def ask_question(request: QuestionRequest):
    response = query_engine.query(request.question)
    return AnswerResponse(
        question=request.question,
        answer=str(response)
    )

# Run with: uvicorn app_basic:app --reload

print("✅ Basic FastAPI app created!")
print("Run: uvicorn app_basic:app --reload")
print("Then visit: http://localhost:8000/docs")

### Test the API

In [ ]:
# Test with requests (after starting the server)
import requests

# Uncomment after starting server
# response = requests.post(
#     "http://localhost:8000/ask",
#     json={"question": "What is FastAPI?"}
# )
# print(response.json())

print("💡 Start the server first: uvicorn app_basic:app --reload")
print("💡 Then test with curl or the /docs interface")

---

## 2. Advanced Request/Response Models

In [ ]:
%%writefile models.py
from pydantic import BaseModel, Field, validator
from typing import Optional, List
from enum import Enum

class ModelType(str, Enum):
    GPT4 = "gpt-4"
    GPT35 = "gpt-3.5-turbo"
    CLAUDE = "claude-3"

class QueryRequest(BaseModel):
    """Request model with validation"""
    question: str = Field(..., min_length=5, max_length=500, 
                          description="The question to answer")
    model: ModelType = Field(ModelType.GPT35, description="LLM to use")
    max_tokens: Optional[int] = Field(100, ge=50, le=1000)
    temperature: Optional[float] = Field(0.7, ge=0.0, le=2.0)
    top_k: Optional[int] = Field(3, ge=1, le=10, 
                                 description="Number of sources to retrieve")
    
    @validator('question')
    def question_not_empty(cls, v):
        if not v.strip():
            raise ValueError('Question cannot be empty')
        return v
    
    class Config:
        json_schema_extra = {
            "example": {
                "question": "What is machine learning?",
                "model": "gpt-3.5-turbo",
                "max_tokens": 150,
                "temperature": 0.7,
                "top_k": 3
            }
        }

class Source(BaseModel):
    """Source document information"""
    text: str
    score: float = Field(..., description="Relevance score")
    metadata: Optional[dict] = None

class QueryResponse(BaseModel):
    """Detailed response with sources"""
    question: str
    answer: str
    model_used: str
    sources: List[Source]
    confidence: Optional[float] = Field(None, ge=0.0, le=1.0)
    tokens_used: Optional[int] = None
    response_time_ms: Optional[float] = None

class ErrorResponse(BaseModel):
    """Error response format"""
    error: str
    detail: Optional[str] = None
    request_id: Optional[str] = None

print("✅ Advanced models created!")

---

## 3. Production API with All Features

In [ ]:
%%writefile app_production.py
from fastapi import FastAPI, HTTPException, Depends, status, Request
from fastapi.responses import StreamingResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from slowapi import Limiter, _rate_limit_exceeded_handler
from slowapi.util import get_remote_address
from slowapi.errors import RateLimitExceeded
import time
import uuid
import logging
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.llms.openai import OpenAI
import os
from dotenv import load_dotenv

# Import our models
from models import QueryRequest, QueryResponse, Source, ErrorResponse

load_dotenv()

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Rate limiting
limiter = Limiter(key_func=get_remote_address)

# Initialize FastAPI with metadata
app = FastAPI(
    title="Production AI Q&A API",
    description="""Enterprise-grade AI-powered question answering API.
    
    Features:
    - Semantic search with RAG
    - Multiple LLM support
    - Streaming responses
    - Rate limiting
    - Authentication ready
    """,
    version="2.0.0",
    contact={
        "name": "AI Team",
        "email": "ai@company.com"
    },
    license_info={
        "name": "MIT",
    }
)

# Add rate limiter
app.state.limiter = limiter
app.add_exception_handler(RateLimitExceeded, _rate_limit_exceeded_handler)

# CORS configuration
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000", "http://localhost:8080"],  # Frontend URLs
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Initialize knowledge base (in production, load from DB)
docs = [
    Document(text="Python is a high-level programming language.", 
             metadata={"source": "doc1", "category": "programming"}),
    Document(text="Machine learning is a subset of artificial intelligence.",
             metadata={"source": "doc2", "category": "ai"}),
    Document(text="FastAPI is a modern Python web framework.",
             metadata={"source": "doc3", "category": "web"})
]

index = VectorStoreIndex.from_documents(docs)

# Health check
@app.get("/health")
async def health_check():
    """Check if the API is running"""
    return {
        "status": "healthy",
        "timestamp": time.time()
    }

# Main query endpoint with rate limiting
@app.post("/query", 
          response_model=QueryResponse,
          responses={
              200: {"model": QueryResponse},
              429: {"model": ErrorResponse, "description": "Too many requests"},
              500: {"model": ErrorResponse, "description": "Internal server error"}
          })
@limiter.limit("10/minute")  # 10 requests per minute per IP
async def query(request: Request, query_request: QueryRequest):
    """Query the AI with rate limiting"""
    request_id = str(uuid.uuid4())
    start_time = time.time()
    
    try:
        logger.info(f"Request {request_id}: {query_request.question}")
        
        # Configure LLM based on request
        Settings.llm = OpenAI(
            model=query_request.model.value,
            temperature=query_request.temperature,
            max_tokens=query_request.max_tokens
        )
        
        # Query
        query_engine = index.as_query_engine(
            similarity_top_k=query_request.top_k
        )
        response = query_engine.query(query_request.question)
        
        # Extract sources
        sources = [
            Source(
                text=node.text[:200],  # Truncate for API
                score=node.score,
                metadata=node.metadata
            )
            for node in response.source_nodes
        ]
        
        response_time = (time.time() - start_time) * 1000
        
        logger.info(f"Request {request_id} completed in {response_time:.2f}ms")
        
        return QueryResponse(
            question=query_request.question,
            answer=str(response),
            model_used=query_request.model.value,
            sources=sources,
            response_time_ms=response_time
        )
        
    except Exception as e:
        logger.error(f"Request {request_id} failed: {str(e)}")
        raise HTTPException(
            status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
            detail=ErrorResponse(
                error="Query failed",
                detail=str(e),
                request_id=request_id
            ).dict()
        )

# Batch endpoint
@app.post("/batch", response_model=List[QueryResponse])
@limiter.limit("5/minute")
async def batch_query(request: Request, queries: List[QueryRequest]):
    """Process multiple queries in batch"""
    if len(queries) > 10:
        raise HTTPException(
            status_code=400,
            detail="Maximum 10 queries per batch"
        )
    
    results = []
    for query_request in queries:
        # Create a mock request object for the limiter
        result = await query(request, query_request)
        results.append(result)
    
    return results

print("✅ Production API created!")
print("Run: uvicorn app_production:app --reload")

---

## 4. Streaming Responses for LLMs

In [ ]:
%%writefile app_streaming.py
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from llama_index.core import VectorStoreIndex, Document
import json
import asyncio

app = FastAPI(title="Streaming AI API")

# Setup index
docs = [
    Document(text="Streaming allows real-time response generation for better UX.")
]
index = VectorStoreIndex.from_documents(docs)

class StreamRequest(BaseModel):
    question: str

@app.post("/stream")
async def stream_query(request: StreamRequest):
    """Stream the response token by token"""
    
    async def generate():
        # Create streaming query engine
        query_engine = index.as_query_engine(streaming=True)
        streaming_response = query_engine.query(request.question)
        
        # Stream each token
        for text in streaming_response.response_gen:
            # Send as Server-Sent Events (SSE) format
            yield f"data: {json.dumps({'token': text})}\n\n"
            await asyncio.sleep(0.01)  # Small delay for demo
        
        # Send completion signal
        yield f"data: {json.dumps({'done': True})}\n\n"
    
    return StreamingResponse(
        generate(),
        media_type="text/event-stream"
    )

print("✅ Streaming API created!")

### Test Streaming with JavaScript

```javascript
// Frontend code to consume streaming API
const response = await fetch('http://localhost:8000/stream', {
  method: 'POST',
  headers: {'Content-Type': 'application/json'},
  body: JSON.stringify({question: 'What is streaming?'})
});

const reader = response.body.getReader();
const decoder = new TextDecoder();

while (true) {
  const {value, done} = await reader.read();
  if (done) break;
  
  const text = decoder.decode(value);
  const lines = text.split('\n');
  
  for (const line of lines) {
    if (line.startsWith('data: ')) {
      const data = JSON.parse(line.slice(6));
      if (data.token) {
        console.log(data.token); // Display token
      }
    }
  }
}
```

---

## 5. WebSocket for Real-Time Chat

In [ ]:
%%writefile app_websocket.py
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from typing import List
import json
from llama_index.core import VectorStoreIndex, Document

app = FastAPI(title="WebSocket Chat API")

# Setup
docs = [Document(text="WebSocket enables bidirectional real-time communication.")]
index = VectorStoreIndex.from_documents(docs)
query_engine = index.as_query_engine()

class ConnectionManager:
    def __init__(self):
        self.active_connections: List[WebSocket] = []
    
    async def connect(self, websocket: WebSocket):
        await websocket.accept()
        self.active_connections.append(websocket)
    
    def disconnect(self, websocket: WebSocket):
        self.active_connections.remove(websocket)
    
    async def send_message(self, message: dict, websocket: WebSocket):
        await websocket.send_text(json.dumps(message))

manager = ConnectionManager()

@app.websocket("/ws/chat")
async def chat_endpoint(websocket: WebSocket):
    await manager.connect(websocket)
    
    await manager.send_message(
        {"type": "system", "message": "Connected to AI chat!"},
        websocket
    )
    
    try:
        while True:
            # Receive question
            data = await websocket.receive_text()
            message = json.loads(data)
            
            if message.get("type") == "question":
                question = message.get("text")
                
                # Send thinking indicator
                await manager.send_message(
                    {"type": "thinking", "message": "Thinking..."},
                    websocket
                )
                
                # Get answer
                response = query_engine.query(question)
                
                # Send answer
                await manager.send_message(
                    {
                        "type": "answer",
                        "question": question,
                        "answer": str(response)
                    },
                    websocket
                )
    
    except WebSocketDisconnect:
        manager.disconnect(websocket)
        print("Client disconnected")

print("✅ WebSocket API created!")

### WebSocket Client Example

```javascript
const ws = new WebSocket('ws://localhost:8000/ws/chat');

ws.onopen = () => {
  console.log('Connected!');
  
  // Send question
  ws.send(JSON.stringify({
    type: 'question',
    text: 'What is WebSocket?'
  }));
};

ws.onmessage = (event) => {
  const data = JSON.parse(event.data);
  console.log('Received:', data);
  
  if (data.type === 'answer') {
    console.log('Answer:', data.answer);
  }
};
```

---

## 6. Authentication and Security

In [ ]:
%%writefile auth.py
from fastapi import Depends, HTTPException, status, Security
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from jose import JWTError, jwt
from datetime import datetime, timedelta
from typing import Optional

# Configuration
SECRET_KEY = "your-secret-key-here"  # In production: os.getenv("SECRET_KEY")
ALGORITHM = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30

# API Key authentication
API_KEYS = {
    "test-key-123": "user1",
    "prod-key-456": "user2"
}

security = HTTPBearer()

def verify_api_key(credentials: HTTPAuthorizationCredentials = Security(security)):
    """Verify API key from Bearer token"""
    api_key = credentials.credentials
    
    if api_key not in API_KEYS:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid API key"
        )
    
    return API_KEYS[api_key]

def create_access_token(data: dict, expires_delta: Optional[timedelta] = None):
    """Create JWT token"""
    to_encode = data.copy()
    
    if expires_delta:
        expire = datetime.utcnow() + expires_delta
    else:
        expire = datetime.utcnow() + timedelta(minutes=15)
    
    to_encode.update({"exp": expire})
    encoded_jwt = jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)
    
    return encoded_jwt

def verify_token(credentials: HTTPAuthorizationCredentials = Security(security)):
    """Verify JWT token"""
    token = credentials.credentials
    
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        username: str = payload.get("sub")
        if username is None:
            raise HTTPException(
                status_code=status.HTTP_401_UNAUTHORIZED,
                detail="Invalid token"
            )
        return username
    except JWTError:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid token"
        )

print("✅ Authentication module created!")

### Using Authentication

In [ ]:
%%writefile app_secure.py
from fastapi import FastAPI, Depends
from auth import verify_api_key, verify_token
from models import QueryRequest, QueryResponse

app = FastAPI(title="Secure AI API")

# Protected endpoint with API key
@app.post("/api/query")
async def secure_query(
    request: QueryRequest,
    username: str = Depends(verify_api_key)
):
    """Protected endpoint requiring API key"""
    # Process query
    return {"message": f"Query processed for user: {username}"}

# Protected endpoint with JWT
@app.post("/api/v2/query")
async def jwt_query(
    request: QueryRequest,
    username: str = Depends(verify_token)
):
    """Protected endpoint requiring JWT token"""
    return {"message": f"Query processed for user: {username}"}

print("✅ Secure API created!")
print("Test with: curl -H 'Authorization: Bearer test-key-123' ...")

---

## 7. Deployment with Docker

In [ ]:
%%writefile Dockerfile
FROM python:3.11-slim

WORKDIR /app

# Install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application
COPY . .

# Expose port
EXPOSE 8000

# Run with gunicorn for production
CMD ["gunicorn", "app_production:app", "-w", "4", "-k", "uvicorn.workers.UvicornWorker", "-b", "0.0.0.0:8000"]

print("✅ Dockerfile created!")

In [ ]:
%%writefile docker-compose.yml
version: '3.8'

services:
  api:
    build: .
    ports:
      - "8000:8000"
    environment:
      - OPENAI_API_KEY=${OPENAI_API_KEY}
      - DATABASE_URL=${DATABASE_URL}
    volumes:
      - ./storage:/app/storage
    restart: unless-stopped
    
  redis:
    image: redis:7-alpine
    ports:
      - "6379:6379"
    restart: unless-stopped

print("✅ Docker Compose file created!")
print("Run: docker-compose up -d")

---

## 🎯 Practice Exercises

### Exercise 1: Enhance Basic API
Add to the basic API:
- Request validation
- Error handling
- Logging
- Health check endpoint

### Exercise 2: Build Chat API
Create a chat API with:
- Conversation history
- Session management
- Message persistence
- Clear history endpoint

### Exercise 3: Add Caching
Implement Redis caching:
- Cache frequent queries
- TTL configuration
- Cache invalidation
- Performance metrics

### Exercise 4: Monitoring
Add monitoring with:
- Prometheus metrics
- Request duration tracking
- Error rate monitoring
- Custom business metrics

### Challenge: Production-Ready API
Build complete system:
- Multiple endpoints (query, batch, feedback)
- Authentication (API key + JWT)
- Rate limiting (per user)
- Caching (Redis)
- Logging (structured)
- Monitoring (Prometheus)
- Docker deployment
- Load testing (Locust)

---

## 📚 Key Takeaways

1. **FastAPI** - Modern, fast, async-first framework
2. **Pydantic** - Request/response validation
3. **Streaming** - Better UX for LLM responses
4. **WebSocket** - Real-time bidirectional communication
5. **Authentication** - API keys or JWT tokens
6. **Rate Limiting** - Protect your API
7. **Docker** - Consistent deployment
8. **Monitoring** - Track performance and errors

---

## 🚀 Next Steps

**AP32 - Docker & Kubernetes**
- Container orchestration
- Scaling strategies
- Service mesh
- Production deployment

**Resources:**
- [FastAPI Documentation](https://fastapi.tiangolo.com/)
- [Uvicorn](https://www.uvicorn.org/)
- [Pydantic](https://docs.pydantic.dev/)

---

*Created as part of the Applied AI module*  
*Last updated: January 2025*